# 00 - Preparación / compilación de bases EPH

**Este es el notebook base del proyecto.** Compila las bases de la EPH descargadas
manualmente del INDEC y subidas a **Google Drive** (carpeta `carga_EPH`): une la base de
**individuos** con la de **hogares** por el código de vínculo (`CODUSU` + `NRO_HOGAR`) y
deja los datasets listos para que **el resto de los notebooks (01-05) los procesen** sin
tener que volver a descargar ni unir las bases.

**Salida (en Google Drive, `carga_EPH/processed/`):**
- `eph_T<Q><YY>.parquet`: **un archivo por trimestre**, ya con individuos+hogares unidos
  y columnas `ANIO`/`TRIMESTRE`. Se guardan en Drive (persistentes entre sesiones). No se
  genera un único panel combinado: desbordaría la RAM de Colab. Los notebooks leen lo que
  necesitan con `load_panel(columns=..., quarters=..., out_dir=PROCESSED_DIR)`.

**Flujo:** Setup (clonar repo + montar Drive + definir carpeta de salida) → Diagnóstico
de `carga_EPH` → Trimestres disponibles → Compilar (1 parquet por trimestre, a Drive) →
Verificación (merge + montos numéricos + quiebre de esquema 4T2023).

**Cómo agregar un trimestre nuevo:**
1. Descargar del INDEC el `.zip` del trimestre (ej. `EPH_usu_4_Trim_2025_txt.zip`),
   que contiene `usu_individual_T<Q><YY>.txt` y `usu_hogar_T<Q><YY>.txt`.
2. Subir el `.zip` (sin descomprimir) a Google Drive, carpeta `carga_EPH`.
3. Volver a correr este notebook (con `overwrite=False` compila solo lo que falte).

> ⚠️ **Quiebre de esquema en 4T2023.** Los trimestres ≤ T3-2023 tienen menos columnas
> (177 individual / 88 hogar) que los ≥ T4-2023 (235 / 98). Las variables nuevas
> (`EMPLEO`, `SECTOR`, ingresos/pensiones desagregados, deciles `P_DECCF`) no existen en
> los trimestres viejos. Ver `.claude/memoria_EPH.md`.

## Setup (Colab)

Clona el repo para tener acceso a `src/data_loader.py`, y monta Google Drive para
leer los `.zip` subidos a `carga_EPH`.

In [ ]:
import sys, os

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Carpeta de salida en Drive (persistente entre sesiones de Colab).
# Acá se guardan los parquets compilados; los notebooks 01-05 los leen de la misma carpeta.
PROCESSED_DIR = "/content/drive/MyDrive/carga_EPH/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Parquets compilados se guardan en:", PROCESSED_DIR)

## Diagnóstico: ¿qué hay en carga_EPH?

Si la celda "Disponibles" más abajo devuelve una lista vacía, correr esta celda
primero para ver qué archivos detecta Colab en la carpeta de Drive y qué contiene
un `.zip` de ejemplo.

In [ ]:
from src.data_loader import DRIVE_DIR
import os, zipfile

print("DRIVE_DIR:", DRIVE_DIR, "| existe:", os.path.isdir(DRIVE_DIR))

if os.path.isdir(DRIVE_DIR):
    archivos = os.listdir(DRIVE_DIR)
    print(f"Archivos encontrados ({len(archivos)}):")
    for a in archivos[:10]:
        print(" -", a)

    zips = [a for a in archivos if a.lower().endswith(".zip")]
    if zips:
        ejemplo = os.path.join(DRIVE_DIR, zips[0])
        print(f"\nContenido de {zips[0]}:")
        with zipfile.ZipFile(ejemplo) as zf:
            for n in zf.namelist():
                print(" -", n)

## Chequeo de disponibilidad

Escanea `/content/drive/MyDrive/carga_EPH` (y `data/raw/` del repo) buscando los `.zip`
del INDEC, y lista los trimestres detectados (con ambas bases, individual y hogar).

## Compilación del panel (memory-safe, guarda en Drive)

Procesa **un trimestre por vez**: une individuos + hogares por `CODUSU` + `NRO_HOGAR`,
agrega `ANIO`/`TRIMESTRE` y guarda **un parquet por trimestre** en `PROCESSED_DIR`
(carpeta de Drive `carga_EPH/processed`, **persistente** entre sesiones).

> No se arma un único DataFrame con los 36 trimestres en memoria (desbordaría la RAM de
> Colab). Los notebooks 01-05 leen los trimestres que necesiten con
> `load_panel(columns=..., quarters=..., out_dir=PROCESSED_DIR)`.

from src.data_loader import build_panel
import pandas as pd

# overwrite=True para regenerar todo. Una vez compilado, en próximas corridas podés
# usar overwrite=False (default) para que saltee los trimestres que ya están en Drive.
resumen = build_panel(available, out_dir=PROCESSED_DIR, overwrite=True)
df_resumen = pd.DataFrame(resumen)
print("Trimestres en resumen:", len(df_resumen))
print("Total de filas compiladas:", df_resumen["filas"].sum())
df_resumen[["year", "period", "filas", "columnas", "estado"]]

In [ ]:
# Verificación del merge: muestra de un trimestre (lee solo columnas clave, memory-safe).
# Verificá que ITF/IPCF sean numéricos (no texto con coma): IPCF debe verse como 2933333.33
from src.data_loader import load_panel

muestra = load_panel(
    columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "ESTADO", "ITF", "IPCF"],
    quarters=[(2025, 4)],
    out_dir=PROCESSED_DIR,
)
print("Shape muestra T4-2025:", muestra.shape)
print("dtype ITF:", muestra["ITF"].dtype, "| dtype IPCF:", muestra["IPCF"].dtype)
muestra.head()

In [ ]:
# Verificación del merge: muestra de un trimestre (lee solo columnas clave, memory-safe)
from src.data_loader import load_panel

muestra = load_panel(
    columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "ESTADO", "ITF", "IPCF"],
    quarters=[(2025, 4)],
)
print("Shape muestra T4-2025:", muestra.shape)
muestra.head()

## Cómo usan estos datos los notebooks 01-05

Cada notebook de análisis monta Drive, clona el repo y llama a `load_panel` apuntando a
la carpeta de Drive, pidiendo solo las columnas (y trimestres) que necesite:

```python
from src.data_loader import load_panel

PROCESSED_DIR = "/content/drive/MyDrive/carga_EPH/processed"

# Ejemplo (demografía): edad, sexo, región y ponderador, todos los trimestres
df = load_panel(columns=["CH06", "CH04", "REGION", "PONDERA"], out_dir=PROCESSED_DIR)

# Ejemplo (informalidad, solo esquema nuevo): restringir a T4-2023 en adelante
df = load_panel(
    columns=["EMPLEO", "SECTOR", "PONDERA"],
    quarters=[(2023, 4), (2024, 1), (2024, 2), (2024, 3), (2024, 4)],
    out_dir=PROCESSED_DIR,
)
```

Los parquets por trimestre quedan en Drive (`carga_EPH/processed`), así que no hay que
recompilar: una vez corrido este notebook 00, los demás leen directo de ahí.

## Resultado

Los parquets compilados quedan en **Google Drive** (`carga_EPH/processed/eph_T<Q><YY>.parquet`),
persistentes entre sesiones. Los notebooks 01-05 los leen desde ahí con `load_panel(..., out_dir=PROCESSED_DIR)`
sin necesidad de volver a compilar.

Para incorporar un trimestre nuevo en el futuro: subir su `.zip` a `carga_EPH` y volver a
correr este notebook con `overwrite=False` (compila solo los trimestres que falten).

## Subir resultados a GitHub (opcional)

Los archivos `.parquet` generados en `data/processed/` quedan disponibles para los demás notebooks vía `raw.githubusercontent.com` una vez que se comiteen y pusheen al repo (esto se hace desde el entorno local, no desde Colab).